In [1]:
# =====================================================
# Symptom → Disease Prediction System
# Feature Selection + Final CatBoost Model
# =====================================================

import pandas as pd
import numpy as np
import os

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score



In [3]:
# =====================================================
# 1. LOAD DATASET
# =====================================================

print("Loading dataset...")

df = pd.read_csv("Datasets/data.csv")

print("Total samples:", len(df))
print("Total features:", len(df.columns) - 1)



Loading dataset...
Total samples: 246945
Total features: 377


In [4]:
# =====================================================
# 2. SPLIT FEATURES AND TARGET
# =====================================================

X = df.drop("diseases", axis=1)
y = df["diseases"]

# Convert to float32 to reduce memory usage
X = X.astype("float32")


In [5]:

# =====================================================
# 3. TRAIN / TEST SPLIT
# =====================================================

print("Splitting dataset...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


Splitting dataset...
Training shape: (197556, 377)
Testing shape: (49389, 377)


In [6]:
# =====================================================
# 4. TRAIN QUICK MODEL FOR FEATURE IMPORTANCE
# =====================================================

print("\nTraining quick model for feature importance...")

importance_model = CatBoostClassifier(
    iterations=40,
    depth=6,
    learning_rate=0.1,
    loss_function="MultiClass",

    verbose=10,

    # Use CPU cores
    thread_count=6,

    # Enable pause / resume
    save_snapshot=True,
    snapshot_file="importance_snapshot",
    snapshot_interval=20
)

importance_model.fit(
    X_train,
    y_train
)


Training quick model for feature importance...


KeyboardInterrupt: 

In [7]:

# =====================================================
# 5. GET FEATURE IMPORTANCE
# =====================================================

print("Calculating symptom importance...")

importances = importance_model.get_feature_importance()

feature_names = X.columns

importance_df = pd.DataFrame({
    "symptom": feature_names,
    "importance": importances
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print("\nTop Important Symptoms:")
print(importance_df.head(15))


Calculating symptom importance...


CatBoostError: Model has no meta information needed to calculate feature importances.                             Pass training dataset to this function.

In [ ]:

# =====================================================
# 6. REMOVE LOW IMPORTANCE SYMPTOMS
# =====================================================

threshold = 0.01

important_features = importance_df[
    importance_df["importance"] > threshold
]["symptom"]

print("\nOriginal symptom count:", X.shape[1])
print("Filtered symptom count:", len(important_features))

# Filter dataset
X_train_filtered = X_train[important_features]
X_test_filtered = X_test[important_features]

# Save selected symptoms
important_features.to_csv("important_symptoms.csv", index=False)

print("Important symptoms saved!")


In [8]:
from sklearn.preprocessing import LabelEncoder
import pickle
# Encode labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
# Save encoder
pickle.dump(encoder, open("models/label_encoder.pkl", "wb"))

# Create symptoms dictionary
symptoms_dict = {
    symptom: idx
    for idx, symptom in enumerate(X.columns)
}

pickle.dump(symptoms_dict, open("models/symptoms_dict.pkl", "wb"))

In [ ]:

# =====================================================
# 7. CREATE CATBOOST POOLS
# =====================================================

train_pool = Pool(X_train_filtered, y_train)
test_pool = Pool(X_test_filtered, y_test)


In [ ]:

# =====================================================
# 8. TRAIN FINAL MODEL
# =====================================================

print("\nTraining final CatBoost model...")

model = CatBoostClassifier(

    iterations=300,
    learning_rate=0.05,
    depth=6,

    loss_function="MultiClass",
    eval_metric="Accuracy",

    thread_count=6,

    verbose=50,

    # Pause / Resume training
    save_snapshot=True,
    snapshot_file="catboost_snapshot",
    snapshot_interval=30
)

model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True
)

print("Training completed!")


In [ ]:

# =====================================================
# 9. SAVE MODEL
# =====================================================

model_path = "models/catboost_disease_model.cbm"

model.save_model(model_path)

print("Model saved successfully!")


In [ ]:

# =====================================================
# 10. EVALUATE MODEL
# =====================================================

print("\nEvaluating model...")

preds = model.predict(X_test_filtered)

accuracy = accuracy_score(y_test, preds)

print("Final Accuracy:", accuracy)


In [ ]:

# =====================================================
# 11. TOP 3 DISEASE PREDICTIONS
# =====================================================

print("\nExample Predictions (Top 3 diseases):")

probs = model.predict_proba(X_test_filtered[:5])

disease_names = model.classes_

for i, p in enumerate(probs):

    top3 = np.argsort(p)[-3:][::-1]

    print(f"\nPatient {i+1}")

    for rank, idx in enumerate(top3):

        disease = disease_names[idx]
        probability = p[idx]

        print(f"{rank+1}. {disease} → {probability:.4f}")

In [ ]:
model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    verbose=100
)

In [ ]:
# ==========================================
# Swasth Disease Prediction Model Trainer
# ==========================================

import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1️⃣ Load Dataset
# ==========================================

print("Loading dataset...")

DATA_PATH = "Datasets/data.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Training.csv not found inside Datasets folder")

df = pd.read_csv(DATA_PATH)

# Remove unnamed columns if present
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Separate features and target
X = df.drop("diseases", axis=1)
y = df["diseases"]

print("Total samples:", len(df))
print("Total symptoms (features):", len(X.columns))
print("Total diseases:", len(y.unique()))

# ==========================================
# 2️⃣ Encode Target Labels
# ==========================================

encoder = LabelEncoder() #It converts categorical text labels into numbers.
y_encoded = encoder.fit_transform(y)

# ==========================================
# 3️⃣ Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

# ==========================================
# 4️⃣ Train SVC Model
# ==========================================

print("Training model...")


from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    auto_class_weights='Balanced',
    verbose=100
)


model.fit(X_train, y_train)

# ==========================================
# 5️⃣ Evaluate Model
# ==========================================

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ==========================================
# 6️⃣ Create Symptoms Dictionary
# ==========================================

# Create mapping of symptom name → index
symptoms_dict = {
    symptom: index
    for index, symptom in enumerate(X.columns)
}

print("\nSymptoms dictionary created.")
print("Total symptoms mapped:", len(symptoms_dict))

# ==========================================
# 7️⃣ Save Model Files
# ==========================================

MODELS_DIR = "models"

if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

# Save trained model
with open(os.path.join(MODELS_DIR, "svc.pkl"), "wb") as f:
    pickle.dump(model, f)

# Save label encoder
with open(os.path.join(MODELS_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(encoder, f)

# Save symptoms dictionary
with open(os.path.join(MODELS_DIR, "symptoms_dict.pkl"), "wb") as f:
    pickle.dump(symptoms_dict, f)

print("\nAll files saved successfully inside /models folder:")
print(" - svc.pkl")
print(" - label_encoder.pkl")
print(" - symptoms_dict.pkl")

print("\nTraining completed successfully.")

Loading dataset...
Total samples: 246945
Total symptoms (features): 377
Total diseases: 713
Training set shape: (197556, 377)
Testing set shape: (49389, 377)
Training model...
0:	learn: 6.4556470	total: 45.4s	remaining: 6h 17m 33s


In [3]:
# ==========================================
# Swasth Disease Prediction Model Trainer (Optimized)
# ==========================================

import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1️⃣ Load Dataset
# ==========================================
print("Loading dataset...")

DATA_PATH = "Datasets/data.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Training.csv not found inside Datasets folder")

df = pd.read_csv(DATA_PATH)

# Remove unnamed columns if present
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Separate features and target
X = df.drop("diseases", axis=1)
y = df["diseases"]

print("Total samples:", len(df))
print("Total symptoms (features):", len(X.columns))
print("Total diseases:", len(y.unique()))

# ==========================================
# 2️⃣ Encode Target Labels
# ==========================================
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# ==========================================
# 3️⃣ Train-Test Split
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

# ==========================================
# 4️⃣ Train Optimized CatBoost Model
# ==========================================
print("Training CatBoost model...")

from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=100,              # reduced for faster training
    learning_rate=0.15,          # slightly higher learning rate
    depth=6,                     # shallower trees for speed
    loss_function='MultiClass',
    auto_class_weights='Balanced',
    thread_count=-1,             # use all CPU cores
    verbose=20,                  # print progress every 20 iterations
    save_snapshot=True,          # enable snapshot for pause/resume
    snapshot_file='models/catboost_snapshot.cbm',
    snapshot_interval=1200,       # save every 10 mins
    early_stopping_rounds=15     # stop if no improvement
)

# Fit the model
model.fit(
    X_train,
    y_train,
    eval_set=(X_test, y_test),
    use_best_model=True
)

# ==========================================
# 5️⃣ Evaluate Model
# ==========================================
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ==========================================
# 6️⃣ Create Symptoms Dictionary
# ==========================================
symptoms_dict = {symptom: idx for idx, symptom in enumerate(X.columns)}

print("\nSymptoms dictionary created.")
print("Total symptoms mapped:", len(symptoms_dict))

# ==========================================
# 7️⃣ Save Model Files
# ==========================================
MODELS_DIR = "models"
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

# Save trained CatBoost model
model.save_model(os.path.join(MODELS_DIR, "catboost_model.cbm"))

# Save label encoder
with open(os.path.join(MODELS_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(encoder, f)

# Save symptoms dictionary
with open(os.path.join(MODELS_DIR, "symptoms_dict.pkl"), "wb") as f:
    pickle.dump(symptoms_dict, f)

print("\nAll files saved successfully inside /models folder:")
print(" - catboost_model.cbm")
print(" - label_encoder.pkl")
print(" - symptoms_dict.pkl")

print("\nTraining completed successfully.")

Loading dataset...
Total samples: 246945
Total symptoms (features): 377
Total diseases: 713
Training set shape: (197556, 377)
Testing set shape: (49389, 377)
Training CatBoost model...
0:	learn: 6.2851531	test: 6.2797539	best: 6.2797539 (0)	total: 42.1s	remaining: 1h 9m 27s
20:	learn: 3.5444970	test: 3.5373636	best: 3.5373636 (20)	total: 21m 44s	remaining: 1h 21m 46s
40:	learn: 2.3234812	test: 2.3459009	best: 2.3459009 (40)	total: 47m 19s	remaining: 1h 8m 5s
60:	learn: 1.7209543	test: 1.7512350	best: 1.7512350 (60)	total: 1h 10m 6s	remaining: 44m 49s
80:	learn: 1.4176352	test: 1.4493035	best: 1.4493035 (80)	total: 1h 32m 20s	remaining: 21m 39s
99:	learn: 1.1731267	test: 1.2140114	best: 1.2140114 (99)	total: 1h 53m 9s	remaining: 0us

bestTest = 1.214011447
bestIteration = 99


Model Accuracy: 75.88 %

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        33
           1       0.27      0.96      0.42        28
  

/Users/paraschandramola/Desktop/Final-Year-Project/swasth/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/paraschandramola/Desktop/Final-Year-Project/swasth/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/paraschandramola/Desktop/Final-Year-Project/swasth/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division`


All files saved successfully inside /models folder:
 - catboost_model.cbm
 - label_encoder.pkl
 - symptoms_dict.pkl

Training completed successfully.


In [4]:
# our accuracy is 75
print(model.score(X_train, y_train))
print(model.score(X_test, y_test))

KeyboardInterrupt: 

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/datasets/ritzboy/sympotms-disease-model'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
import pandas as pd
import numpy as np
import os

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
# =====================================================
# 1. LOAD DATASET
# =====================================================

print("Loading dataset...")

df = pd.read_csv("/kaggle/input/datasets/ritzboy/sympotms-disease-model/data.csv")

print("Total samples:", len(df))
print("Total features:", len(df.columns) - 1)

# =====================================================
# 2. SPLIT FEATURES AND TARGET
# =====================================================

X = df.drop("diseases", axis=1)
y = df["diseases"]

# Convert to float32 to reduce memory usage
X = X.astype("float32")
# =====================================================
# 1. TRAIN / TEST SPLIT
# =====================================================

print("Splitting dataset...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

# =====================================================
# 2. CREATE POOLS
# =====================================================

train_pool = Pool(X_train, y_train)
test_pool = Pool(X_test, y_test)




# =====================================================
# 3. MODEL CONFIGURATION
# =====================================================

print("\nInitializing CatBoost model...")


print("Training on GPU...")

model = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.03,
    depth=8,

    loss_function="MultiClass",
    eval_metric="Accuracy",

    task_type="GPU",      # 🔥 ENABLE GPU
    devices="0",          # Use first GPU

    auto_class_weights="Balanced",

    early_stopping_rounds=100,

    verbose=100
)



# =====================================================
# 4. TRAIN MODEL
# =====================================================

print("\nTraining model...")

model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True
)



print("Training completed!")


# =====================================================
# 5. EVALUATE MODEL
# =====================================================

print("\nEvaluating model...")

preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)

print("Test Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, preds))

# =====================================================
# 6. PREDICTION CONFIDENCE CHECK
# =====================================================

print("\nChecking prediction confidence...")

probs = model.predict_proba(X_test)

print("Example prediction probabilities:")
print(probs[:5])


# =====================================================
# 9. SAVE MODEL
# =====================================================

# model_path = "models/catboost_disease_model.cbm"


model.save_model("/kaggle/working/disease_prediction_model.cbm")

print("Model saved successfully!")


Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/ritzboy/sympotms-disease-model/data.csv'

In [ ]:

# =====================================================
# 2. CREATE POOLS
# =====================================================

train_pool = Pool(X_train, y_train)
test_pool = Pool(X_test, y_test)